Embedding the photos (3-5pic per person) into vector then calculate the average then store into the csv

In [8]:
import pandas as pd
import numpy as np
import cv2
# Use FaceNet class from the dedicated package for reliable model loading
from keras_facenet import FaceNet 
from mtcnn.mtcnn import MTCNN

In [9]:
STUDENT_CSV = 'student_csv/students.csv'
EMBEDDED_CSV = 'student_csv/student_with_embeddings.csv'
TARGET_SIZE = (160, 160) # FaceNet's input size is 160x160

# Initialize the MTCNN detector once
detector = MTCNN()
embedder = FaceNet()

Exception ignored in: <_io.BufferedReader>
Traceback (most recent call last):
  File "c:\Users\enyee\AppData\Local\Programs\Python\Python313\Lib\site-packages\lz4\frame\__init__.py", line 753, in flush
    self._fp.flush()
ValueError: I/O operation on closed file.
Exception ignored in: <_io.BufferedReader>
Traceback (most recent call last):
  File "c:\Users\enyee\AppData\Local\Programs\Python\Python313\Lib\site-packages\lz4\frame\__init__.py", line 753, in flush
    self._fp.flush()
ValueError: I/O operation on closed file.
Exception ignored in: <_io.BufferedReader>
Traceback (most recent call last):
  File "c:\Users\enyee\AppData\Local\Programs\Python\Python313\Lib\site-packages\lz4\frame\__init__.py", line 753, in flush
    self._fp.flush()
ValueError: I/O operation on closed file.


In [10]:
# for the student.csv
def extract_face(filename, required_size=TARGET_SIZE):
    image = cv2.imread(filename)
    if image is None:
        return None

    results = detector.detect_faces(image)
    if not results:
        return None

    x1, y1, width, height = results[0]['box']
    x1, y1 = abs(x1), abs(y1)
    x2, y2 = x1 + width, y1 + height

    face = image[y1:y2, x1:x2]
    face = cv2.resize(face, required_size)
    face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)

    return face

def get_embedding(face_pixels):
    face_pixels = np.expand_dims(face_pixels, axis=0)
    yhat = embedder.embeddings(face_pixels)
    return yhat[0]


# generate average embedding vector for student photos
def generate_student_embeddings():
    df = pd.read_csv(STUDENT_CSV)
    final_embeddings = []

    print("\nGenerating embeddings for students...\n")

    for index, row in df.iterrows():
        img_paths = [row['img1'], row['img2'], row['img3'], row['img4'], row['img5']]
        person_embeddings = []

        for img in img_paths:
            if pd.isna(img) or img == "":
                continue

            face = extract_face(img)
            if face is not None:
                emb = get_embedding(face)
                person_embeddings.append(emb)

        if person_embeddings:
            avg_embedding = np.mean(person_embeddings, axis=0)
            avg_embedding = ','.join(map(str, avg_embedding))
        else:
            avg_embedding = ''

        final_embeddings.append(avg_embedding)
        print(f"✅ Processed: {row['student_name']}")

    df['embedding'] = final_embeddings
    df.to_csv(EMBEDDED_CSV, index=False)
    print("\n✅ Finished! Saved to:", EMBEDDED_CSV)

In [11]:
generate_student_embeddings()


Generating embeddings for students...

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
✅ Processed: TEOH XIN YAO
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
✅ Processed: LEONG YUAN QI
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
✅ Processed: TEOW WEN TING
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
✅ Processed: CHENG EN YEE

✅ Finished! Saved to: student_csv/student_with_embeddings.csv


In [12]:
# TENSORFLOW MASK DETECTION PREDICTION 
import cv2
from mtcnn import MTCNN
import numpy as np

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import img_to_array

# Initialize the MTCNN face detector
detector = MTCNN()

#mask_model = load_model("mask_detector.h5")   
mask_model = load_model("mask_detectorV2.h5")   # load it
IMG_SIZE = 224
CLASS_LABELS = ["Mask", "No Mask"]


Exception ignored in: <_io.BufferedReader>
Traceback (most recent call last):
  File "c:\Users\enyee\AppData\Local\Programs\Python\Python313\Lib\site-packages\lz4\frame\__init__.py", line 753, in flush
    self._fp.flush()
ValueError: I/O operation on closed file.
Exception ignored in: <_io.BufferedReader>
Traceback (most recent call last):
  File "c:\Users\enyee\AppData\Local\Programs\Python\Python313\Lib\site-packages\lz4\frame\__init__.py", line 753, in flush
    self._fp.flush()
ValueError: I/O operation on closed file.
Exception ignored in: <_io.BufferedReader>
Traceback (most recent call last):
  File "c:\Users\enyee\AppData\Local\Programs\Python\Python313\Lib\site-packages\lz4\frame\__init__.py", line 753, in flush
    self._fp.flush()
ValueError: I/O operation on closed file.


In [13]:
from scipy.spatial.distance import euclidean
import pandas as pd
import numpy as np

THRESHOLD = 0.8

# Load database
df = pd.read_csv("student_csv/student_with_embeddings.csv")

db_embeddings = []
for emb in df['embedding']:
    if pd.isna(emb) or emb == "":
        db_embeddings.append(None)
    else:
        db_embeddings.append(np.array(list(map(float, emb.split(',')))))

In [14]:
def extract_face(frame, required_size=TARGET_SIZE):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = detector.detect_faces(rgb)

    if not results:
        return None, None

    x, y, w, h = results[0]['box']
    x, y = abs(x), abs(y)

    face = frame[y:y+h, x:x+w]
    if face.size == 0:
        return None, None

    face = cv2.resize(face, required_size)
    face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)

    return face, (x, y, w, h)


def get_embedding(face_pixels):
    face_pixels = np.expand_dims(face_pixels, axis=0)
    return embedder.embeddings(face_pixels)[0]


def recognize_face(live_embedding):
    min_dist = float("inf")
    identity_index = -1

    for i, db_emb in enumerate(db_embeddings):
        if db_emb is None:
            continue

        dist = euclidean(live_embedding, db_emb)

        if dist < min_dist:
            min_dist = dist
            identity_index = i

    if min_dist < THRESHOLD:
        return df.loc[identity_index]
    else:
        return None

In [15]:
# cap = cv2.VideoCapture(0)

# if not cap.isOpened():
#     print("Error: Cannot open webcam")

In [16]:
# cap = cv2.VideoCapture(0)

# if not cap.isOpened():
#     print("Error: Cannot open webcam")

# # Main loop for face detection
# while True:
#     ret, frame = cap.read()
#     if not ret:
#         print("Failed to grab frame")
#         break

#     # Convert to RGB (MTCNN expects RGB)
#     rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

#     # Detect faces
#     faces = detector.detect_faces(rgb_frame)

#     # Draw bounding boxes and crop face
#     cropped_face = None


#     if len(faces) == 0:
#         cv2.putText(frame, "No face detected", (20, 40),
#                     cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

#     elif len(faces) > 1:
#         cv2.putText(frame, "Multiple faces detected! Error!", (20, 40),
#                     cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

#     else:
#         # Exactly 1 face detected
#         face = faces[0]
#         x, y, w, h = face['box']

#         # Ensure coordinates are valid (no negative values)
#         x = max(0, x)
#         y = max(0, y)

#         # Draw bounding box
#         cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

#         # Crop the face region
#         cropped_face = frame[y:y+h, x:x+w]

#         if cropped_face is not None and cropped_face.size != 0:
#             # crop face → classify
#             face_resized = cv2.resize(cropped_face, (IMG_SIZE, IMG_SIZE))
#             face_resized = img_to_array(face_resized)
#             face_resized = face_resized / 255.0
#             face_resized = np.expand_dims(face_resized, axis=0)
            
#             prediction = mask_model.predict(face_resized, verbose=0)
#             label_index = np.argmax(prediction)
#             label = CLASS_LABELS[label_index]

#             if label == "Mask":
#                 cv2.putText(frame, "Take off mask", (x, y - 10),
#                         cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)
#             else:
#                 cv2.putText(frame, "No Mask", (x, y - 10),
#                         cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
#                 #add face recognition code here (only proceed if no mask)
#                 face_pixels, box = extract_face(frame)

#                 if face_pixels is not None:
#                     live_embedding = get_embedding(face_pixels)
#                     match = recognize_face(live_embedding)

#                     if match is not None:
#                         cv2.putText(frame, f"ID: {match['student_id']}", 
#                                     (10, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,0), 2)

#                         cv2.putText(frame, f"Name: {match['student_name']}", 
#                                     (10, 110), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,0), 2)

#                         cv2.putText(frame, f"Programme: {match['programme']}", 
#                                     (10, 140), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,0), 2)

#                         cv2.putText(frame, f"Faculty: {match['faculty']}", 
#                                     (10, 170), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,0), 2)
#                     else:
#                         cv2.putText(frame, "Unknown Face", 
#                                     (10, 80), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)
    
#     # Show the main webcam frame
#     cv2.imshow("Webcam Face Detection", frame)

#     # Exit if user presses 'q'
#     if cv2.waitKey(1) & 0xFF == ord('q'):
#         break

# # Release resources
# cap.release()
# cv2.destroyAllWindows()